In [ ]:
from google.colab import drive
drive.mount("/content/drive")

CARPETA = "/content/drive/MyDrive/fresas"
R = "/content/recortes"
PRUEBA = "/content/prueba"
!unzip -q -o "$CARPETA/recortes.zip" -d /content
!unzip -q -o "$CARPETA/fotos_prueba.zip" -d $PRUEBA
!cp "$CARPETA/etiquetas_propias.csv" "$CARPETA/clasificador.pt" "$CARPETA/clasificador.json" /content/

In [ ]:
import json
import torch
import torch.nn as nn
from torchvision import transforms, models
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, SamModel, SamProcessor

disp = "cuda" if torch.cuda.is_available() else "cpu"
dino_proc = AutoProcessor.from_pretrained("IDEA-Research/grounding-dino-tiny")
dino = AutoModelForZeroShotObjectDetection.from_pretrained("IDEA-Research/grounding-dino-tiny").to(disp).eval()
sam_proc = SamProcessor.from_pretrained("facebook/sam-vit-base")
sam = SamModel.from_pretrained("facebook/sam-vit-base").to(disp).eval()

cfg = json.load(open("/content/clasificador.json"))
clasif = models.efficientnet_b0()
clasif.classifier[1] = nn.Linear(clasif.classifier[1].in_features, 2)
clasif.load_state_dict(torch.load("/content/clasificador.pt", map_location=disp))
clasif = clasif.to(disp).eval()
tf = transforms.Compose([transforms.Resize((cfg["tamano"], cfg["tamano"])), transforms.ToTensor(),
                         transforms.Normalize(cfg["media"], cfg["desviacion"])])
print("Listo en", disp, "con", cfg["experimento"])

In [ ]:
import time
import numpy as np
import pandas as pd
from PIL import Image

TEXTO = "a red strawberry."
UMBRAL = 0.3
MARGEN = 0.10
LADO = 256


def detectar(img):
    ent = dino_proc(images=img, text=TEXTO, return_tensors="pt").to(disp)
    with torch.no_grad():
        sal = dino(**ent)
    try:
        res = dino_proc.post_process_grounded_object_detection(
            sal, ent.input_ids, threshold=UMBRAL, text_threshold=UMBRAL, target_sizes=[img.size[::-1]])[0]
    except TypeError:
        res = dino_proc.post_process_grounded_object_detection(
            sal, ent.input_ids, box_threshold=UMBRAL, text_threshold=UMBRAL, target_sizes=[img.size[::-1]])[0]
    return res["boxes"].cpu().tolist(), res["scores"].cpu().tolist()


def segmentar(img, cajas):
    ent = sam_proc(img, input_boxes=[cajas], return_tensors="pt").to(disp)
    with torch.no_grad():
        sal = sam(**ent, multimask_output=False)
    return sam_proc.image_processor.post_process_masks(
        sal.pred_masks.cpu(), ent["original_sizes"].cpu(), ent["reshaped_input_sizes"].cpu())[0][:, 0].numpy()


def iou_cajas(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    area = lambda c: (c[2] - c[0]) * (c[3] - c[1])
    return inter / (area(a) + area(b) - inter + 1e-9)


def limpiar_cajas(cajas, puntajes):
    orden = sorted(range(len(cajas)), key=lambda i: -puntajes[i])
    def dentro(i, j):
        a, b = cajas[i], cajas[j]
        ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
        return ix * iy / ((a[2] - a[0]) * (a[3] - a[1]) + 1e-9)
    def centro_en(j, i):
        x0, y0, x1, y1 = cajas[i]
        return x0 <= (cajas[j][0] + cajas[j][2]) / 2 <= x1 and y0 <= (cajas[j][1] + cajas[j][3]) / 2 <= y1
    quedan = []
    for i in orden:
        if all(iou_cajas(cajas[i], cajas[j]) < 0.5 for j in quedan):
            quedan.append(i)
    quedan = [i for i in quedan if sum(centro_en(j, i) for j in quedan if j != i) < 2]
    return [i for k, i in enumerate(quedan) if all(dentro(i, j) < 0.8 for j in quedan[:k])]


def recorte_caja(img, caja):
    x0, y0, x1, y1 = caja
    mx, my = (x1 - x0) * MARGEN, (y1 - y0) * MARGEN
    r = img.crop((max(0, int(x0 - mx)), max(0, int(y0 - my)),
                  min(img.width, int(x1 + mx) + 1), min(img.height, int(y1 + my) + 1)))
    r.thumbnail((LADO, LADO))
    return r


def prob_danada(recortes):
    if not recortes:
        return np.array([])
    ps = []
    with torch.no_grad():
        for i in range(0, len(recortes), 32):
            x = torch.stack([tf(r) for r in recortes[i:i + 32]]).to(disp)
            ps.append(torch.softmax(clasif(x), 1)[:, 1].cpu())
    return torch.cat(ps).numpy()


def rle_a_mascara(s, h, w):
    cnts, p = [], 0
    while p < len(s):
        x = k = 0
        mas = 1
        while mas:
            c = ord(s[p]) - 48
            x |= (c & 0x1f) << 5 * k
            mas = c & 0x20
            p += 1
            k += 1
            if not mas and (c & 0x10):
                x |= -1 << 5 * k
        if len(cnts) > 2:
            x += cnts[-2]
        cnts.append(x)
    plano = np.repeat(np.arange(len(cnts)) % 2, cnts).astype(bool)
    return plano.reshape(w, h).T


def iou_mascaras(a, b):
    union = np.logical_or(a, b).sum()
    return np.logical_and(a, b).sum() / union if union else 0.0


def emparejar(reales, predichas, minimo=0.5):
    pares = sorted(((iou_mascaras(a, b), i, j) for i, a in enumerate(reales) for j, b in enumerate(predichas)), reverse=True)
    usados_r, usados_p, salida = set(), set(), []
    for iou, i, j in pares:
        if iou >= minimo and i not in usados_r and j not in usados_p:
            usados_r.add(i); usados_p.add(j); salida.append((i, j, iou))
    return salida


uno = np.zeros((4, 4), bool); uno[:2, :2] = True
otro = np.zeros((4, 4), bool); otro[:2, :] = True
assert iou_mascaras(uno, otro) == 0.5
assert emparejar([uno, otro], [otro]) == [(1, 0, 1.0)]
assert rle_a_mascara("04", 2, 2).tolist() == [[True, True], [True, True]]
print("Funciones listas")

In [ ]:
eti = pd.read_csv("/content/etiquetas_propias.csv")
val = eti[eti.grupo.isin(["escena02", "escena07", "escena11"]) & eti.etiqueta.isin(["sana", "danada"])]
p_val = prob_danada([Image.open(f"{R}/caja/{r}").convert("RGB") for r in val.recorte])
y_val = (val.etiqueta == "danada").values

opciones = []
for u in [0.5, 0.2, 0.1, 0.05, 0.02, 0.01]:
    pred = p_val >= u
    vp, fp, fn = (pred & y_val).sum(), (pred & ~y_val).sum(), (~pred & y_val).sum()
    f1 = 2 * vp / (2 * vp + fp + fn)
    opciones.append((f1, u))
    print(f"umbral {u}: dañadas encontradas {vp} de {y_val.sum()}, sanas marcadas como dañadas {fp}, F1 {f1:.3f}")
UMBRAL_CLASE = max(opciones)[1]
print("Umbral escogido con la validación de nuestras fotos:", UMBRAL_CLASE)

In [ ]:
import os
SALIDA = f"{CARPETA}/evaluacion"
os.makedirs(SALIDA, exist_ok=True)
coco = json.load(open(f"{PRUEBA}/anotaciones_prueba.json"))
nombre_clase = {c["id"]: c["name"] for c in coco["categories"]}
COLOR = {"sana": np.array([0, 255, 0]), "danada": np.array([255, 0, 255])}


def borde(m):
    interior = m.copy()
    interior[1:-1, 1:-1] = m[1:-1, 1:-1] & m[:-2, 1:-1] & m[2:, 1:-1] & m[1:-1, :-2] & m[1:-1, 2:]
    return m & ~interior


instancias, por_foto = [], []
for im in coco["images"]:
    img = Image.open(f"{PRUEBA}/{im['file_name']}").convert("RGB")
    anot = [a for a in coco["annotations"] if a["image_id"] == im["id"]]
    m_real = [rle_a_mascara(a["segmentation"]["counts"], im["height"], im["width"]) for a in anot]
    c_real = [nombre_clase[a["category_id"]] for a in anot]

    t0 = time.time()
    cajas, puntajes = detectar(img)
    cajas = [cajas[i] for i in limpiar_cajas(cajas, puntajes)] if cajas else []
    m_pred = list(segmentar(img, cajas)) if cajas else []
    p = prob_danada([recorte_caja(img, c) for c in cajas])
    segundos = time.time() - t0
    c_pred = ["danada" if x >= UMBRAL_CLASE else "sana" for x in p]

    pares = emparejar(m_real, m_pred)
    real_ok = {i: (j, iou) for i, j, iou in pares}
    pred_ok = {j for _, j, _ in pares}
    foto = im["file_name"]
    for i, c in enumerate(c_real):
        j, iou = real_ok.get(i, (None, 0.0))
        instancias.append({"foto": foto, "real": c, "predicha": c_pred[j] if j is not None else "no_encontrada",
                           "iou": iou, "prob_danada": p[j] if j is not None else None})
    for j, c in enumerate(c_pred):
        if j not in pred_ok:
            instancias.append({"foto": foto, "real": "no_es_fresa_real", "predicha": c, "iou": 0.0, "prob_danada": p[j]})
    por_foto.append({"foto": foto, "sanas_reales": c_real.count("sana"), "danadas_reales": c_real.count("danada"),
                     "sanas_predichas": c_pred.count("sana"), "danadas_predichas": c_pred.count("danada"),
                     "segundos": segundos})

    arr = np.array(img).astype(float)
    for m, c in zip(m_pred, c_pred):
        arr[m] = arr[m] * 0.55 + COLOR[c] * 0.45
    for i, m in enumerate(m_real):
        if i not in real_ok:
            arr[borde(m)] = [255, 255, 0]
    dib = Image.fromarray(arr.astype(np.uint8))
    dib.thumbnail((1600, 1600))
    dib.save(f"{SALIDA}/{foto.replace('/', '_')}", quality=88)
    print(foto, f"{len(c_real)} reales, {len(c_pred)} encontradas, {len(pares)} emparejadas, {segundos:.1f} s")

ins = pd.DataFrame(instancias)
fot = pd.DataFrame(por_foto)
ins.to_csv(f"{SALIDA}/instancias.csv", index=False)
fot.to_csv(f"{SALIDA}/por_foto.csv", index=False)

In [ ]:
reales = ins[ins.real != "no_es_fresa_real"]
emparejadas = reales[reales.predicha != "no_encontrada"]
vp = len(emparejadas)
fp = (ins.real == "no_es_fresa_real").sum()
fn = (reales.predicha == "no_encontrada").sum()
print("Separación de fresas (IoU de máscara de 0.5 o más)")
print(f"  fresas reales {len(reales)}, encontradas {vp}, no encontradas {fn}, detecciones que no eran fresa o sobraban {fp}")
print(f"  precisión {vp / (vp + fp):.3f}, recall {vp / (vp + fn):.3f}, F1 {2 * vp / (2 * vp + fp + fn):.3f}")
print(f"  IoU promedio de las encontradas {emparejadas.iou.mean():.3f}")
print()
print("Clasificación de las fresas encontradas (filas real, columnas predicha)")
print(pd.crosstab(emparejadas.real, emparejadas.predicha))
y = emparejadas.real == "danada"
pr = emparejadas.predicha == "danada"
d_vp, d_fp, d_fn = (y & pr).sum(), (~y & pr).sum(), (y & ~pr).sum()
print(f"  exactitud {(y == pr).mean():.3f}, dañadas: precisión {d_vp / max(1, d_vp + d_fp):.3f}, recall {d_vp / max(1, d_vp + d_fn):.3f}")
print()
print("Programa completo, dañadas bien separadas y bien clasificadas")
total_d = (reales.real == "danada").sum()
pred_d = (ins.predicha == "danada").sum()
print(f"  dañadas reales {total_d}, encontradas y marcadas dañadas {d_vp}, marcadas dañadas en total {pred_d}")
print()
print("Conteo por foto")
print(f"  error absoluto medio en sanas {(fot.sanas_reales - fot.sanas_predichas).abs().mean():.2f}, "
      f"en dañadas {(fot.danadas_reales - fot.danadas_predichas).abs().mean():.2f}")
print(f"  total real {fot.sanas_reales.sum()} sanas y {fot.danadas_reales.sum()} dañadas, "
      f"total contado {fot.sanas_predichas.sum()} sanas y {fot.danadas_predichas.sum()} dañadas")
print(f"  segundos por foto: promedio {fot.segundos.mean():.2f}, máximo {fot.segundos.max():.2f}")
print()
print(fot.to_string(index=False))